# Factoring 21 with generated modular multiplication

This notebook uses the reusable package in this repository. Run `python -m pip install -e '.[dev]'` from the repository root first, and select that Python environment as the kernel. Every executable cell runs locally without an API key.

The multiplier accepts any small modulus and coprime base. It enumerates a basis-state permutation, so construction is exponential in the input bit length. This is a correctness baseline for small instances; an efficient arithmetic implementation is the next step.

**Hardware follow-up:** The [README](../README.md) now includes successful IBM Fez results for 15 and 21 using smaller compiled circuits. This notebook retains the generic permutation construction as the correctness reference.

In [1]:
from fractions import Fraction
from math import gcd

import numpy as np
import qiskit
from qiskit import transpile
from qiskit.quantum_info import Operator
from shors_algorithm import (
    controlled_modular_multiplication,
    order_finding_circuit,
    recover_periods,
    simulate,
)

print("Qiskit", qiskit.__version__)

Qiskit 2.6.2


In [2]:
N = 21
a = 2
work_qubits = (N - 1).bit_length()
counting_qubits = 2 * work_qubits
shots = 2048
seed = 7

print(f"N={N}, a={a}, gcd(a,N)={gcd(a, N)}")
print(f"{counting_qubits} counting + {work_qubits} work qubits")

N=21, a=2, gcd(a,N)=1
10 counting + 5 work qubits


## The full reversible map

For $x<N$, the gate maps $|x\rangle$ to $|ax \bmod N\rangle$. For $x\ge N$, it fixes $|x\rangle$. With $N=21$, the five work qubits also represent states 21 through 31, so those states need a well-defined action.

The leading control is qubit 0; work bits follow in little-endian order. The combined basis-state index is therefore `2*x + control`. The check below compares amplitudes, including relative phases, for **every** basis state and both control values.

In [3]:
multiplier = controlled_modular_multiplication(a, N)
matrix = Operator(multiplier).data
expected = np.zeros_like(matrix)
for x in range(1 << work_qubits):
    for control in (0, 1):
        y = a * x % N if control and x < N else x
        expected[2 * y + control, 2 * x + control] = 1
np.testing.assert_allclose(matrix, expected, atol=1e-10)
print(f"Verified all {len(matrix)} input states, including both control values.")
print(f"Controlled multiplier: {len(multiplier.definition)} multi-controlled X gates")

Verified all 64 input states, including both control values.
Controlled multiplier: 57 multi-controlled X gates


## Quantum phase estimation

The work register starts at $|1\rangle$. Counting bit $j$ controls multiplication by $a^{2^j} \bmod N$, computed by classical repeated squaring. Each constant multiplier is synthesized directly. The inverse QFT includes the bit-order reversal.

For base 2 modulo 21, the order is 6, so ideal measured phases cluster near $0,1/6,2/6,3/6,4/6,5/6$. The circuit builder takes only the base, modulus, and register size; it is not passed the order or factors.

In [4]:
qc = order_finding_circuit(a, N, counting_qubits)
print(qc.draw(output="text", fold=110))

            ┌───┐┌─────────────────┐                                                          »
counting_0: ┤ H ├┤0                ├──────────────────────────────────────────────────────────»
            ├───┤│                 │┌─────────────────┐                                       »
counting_1: ┤ H ├┤                 ├┤0                ├───────────────────────────────────────»
            ├───┤│                 ││                 │┌──────────────────┐                   »
counting_2: ┤ H ├┤                 ├┤                 ├┤0                 ├───────────────────»
            ├───┤│                 ││                 ││                  │┌─────────────────┐»
counting_3: ┤ H ├┤                 ├┤                 ├┤                  ├┤0                ├»
            ├───┤│                 ││                 ││                  ││                 │»
counting_4: ┤ H ├┤                 ├┤                 ├┤                  ├┤                 ├»
            ├───┤│                 ││   

In [5]:
result = simulate(a, N, counting_qubits=counting_qubits, shots=shots, seed=seed)
print("Verified periods:", result.periods)
print("Factors:", result.factors)
print("Total shots:", sum(result.counts.values()))

Verified periods: (6,)
Factors: (3, 7)
Total shots: 2048


## Interpret measurements, not frequencies

A bitstring represents the integer $y$ and phase $y/2^t$. Its count is just the number of times it appeared. The exact rational continued-fraction calculation checks candidate periods with `pow(a, r, N) == 1`. Reduced fractions can need combination: denominators 2 and 3 together suggest 6.

Periods can be unhelpful for factoring: odd periods and $a^{r/2}=\pm1 \bmod N$ are rejected. The experiment returns `None` when it cannot recover factors, rather than running an unbounded retry loop.

In [6]:
print("bitstring    shots       phase     periods from this outcome alone")
for bitstring, count in sorted(result.counts.items(), key=lambda item: item[1], reverse=True)[:10]:
    phase = Fraction(int(bitstring, 2), 1 << counting_qubits)
    periods = recover_periods({bitstring: count}, a, N, counting_qubits)
    print(f"{bitstring}  {count:6}  {float(phase):10.6f}     {periods}")

if result.factors is not None:
    p, q = result.factors
    assert 1 < p < N and p * q == N
    print(f"\n{N} = {p} * {q}")
else:
    print("Try more shots or a different coprime base.")

bitstring    shots       phase     periods from this outcome alone
0000000000     334    0.000000     ()
1000000000     329    0.500000     ()
1010101011     248    0.666992     ()
0010101011     238    0.166992     (6,)
1101010101     234    0.833008     (6,)
0101010101     231    0.333008     ()
0010101010      73    0.166016     (6,)
0101010110      61    0.333984     ()
1010101010      51    0.666016     ()
1101010110      50    0.833984     (6,)

21 = 3 * 7


## Circuit cost before hardware routing

These counts use an all-to-all `u`/`cx` gate basis. They are not device-specific estimates. The generic synthesis is much larger than the original hand-optimized modulo-15 circuit. For this default 21 example, Qiskit 2.6.2 produces 13,797 CX gates at optimization level 1.

Enough qubits alone does not make such a deep circuit practical on a noisy QPU. The next improvement should be modular arithmetic and synthesis optimization, followed by target-specific compilation and noise analysis.

In [7]:
compiled = transpile(qc, basis_gates=["u", "cx"], optimization_level=1, seed_transpiler=7)
print("Logical qubits:", qc.num_qubits)
print("CX gates:", compiled.count_ops().get("cx", 0))
print("Two-qubit depth:", compiled.depth(lambda instruction: len(instruction.qubits) == 2))

Logical qubits: 15
CX gates: 13797
Two-qubit depth: 13726


## Regression: the original modulus

This uses the same generated multiplier and post-processing with $N=15$, rather than the historical `c_amod15` helper.

In [8]:
original = simulate(a=7, modulus=15, shots=1024, seed=seed)
assert original.factors == (3, 5)
print("N=15:", original.periods, original.factors)

N=15: (4,) (3, 5)


## IBM Quantum hardware results

Both 15 and 21 have now been factored on IBM Fez with smaller compiled circuits. See the [experiment report](../results/ibm_fez_2026-11-16/REPORT.md) for raw-count verification and the [README](../README.md#ibm-hardware) for reproduction commands. This notebook remains a local reference and does not submit hardware jobs.

For a scalable construction, see [Beauregard](https://arxiv.org/abs/quant-ph/0205095).